# Run v5 Bidirectional Hierarchical Stage-1 + PRA-AOG

Every executable cell begins with `!`. This notebook uses the local paths from the previous runs and executes the v5 branch `pra-aog-v5-bidir-stage1`.


In [ ]:
!python -c 'from pathlib import Path; p=Path("/tmp/bidir_hier_pra_aog_v5.env"); p.write_text("export WORKSPACE=\"/home/dfli/instance_slot_aog\"\nexport REPO_DIR=\"$WORKSPACE/clean_v18_v39_v42\"\nexport PARTIMAGENET_ROOT=\"/home/dfli/full_hyco/PartImageNet\"\nexport CONFIG=\"$REPO_DIR/configs/notebook_stage1.yaml\"\nexport BASE_STAGE1_CKPT=\"$REPO_DIR/runs/stage1_quality_upgrade/checkpoints/stage1_best.pt\"\nexport BIDIR_STAGE1_DIR=\"$REPO_DIR/runs/bidir_hier_stage1_v5\"\nexport BIDIR_STAGE1_CKPT=\"$BIDIR_STAGE1_DIR/checkpoints/hier_stage1_best.pt\"\nexport BIDIR_CACHE_DIR=\"$REPO_DIR/artifacts/bidir_hier_stage1_v5_strict_aog\"\nexport TRAIN_CACHE=\"$BIDIR_CACHE_DIR/train_strict_aog_terminals.pt\"\nexport VAL_CACHE=\"$BIDIR_CACHE_DIR/val_strict_aog_terminals.pt\"\nexport HIER_BUNDLE_DIR=\"$REPO_DIR/artifacts/hier_pra_aog_v5_bidir\"\nexport HIER_BUNDLE=\"$HIER_BUNDLE_DIR/hier_pra_aog_v5_bidir_bundle.pt\"\nexport HIER_RUN_DIR=\"$REPO_DIR/runs/hier_pra_aog_v5_bidir\"\nexport HIER_BEST_CKPT=\"$HIER_RUN_DIR/checkpoints/strict_aog_best.pt\"\nexport INFER_DIR=\"$HIER_RUN_DIR/inference\"\nexport PACKAGE_OUT=\"$WORKSPACE/bidir_hier_pra_aog_v5_results.tar.gz\"\n", encoding="utf-8"); print(p.read_text())'

## 1. Checkout branch and install


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && test -d "$REPO_DIR/.git" && git -C "$REPO_DIR" fetch origin pra-aog-v5-bidir-stage1 && (git -C "$REPO_DIR" switch pra-aog-v5-bidir-stage1 || git -C "$REPO_DIR" switch --track -c pra-aog-v5-bidir-stage1 origin/pra-aog-v5-bidir-stage1) && git -C "$REPO_DIR" pull --ff-only origin pra-aog-v5-bidir-stage1 && git -C "$REPO_DIR" log -1 --oneline

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python -m pip install -e ".[dev,vision]" && python -c 'import torch; from partcat_hkg.models import HierarchicalPartCATHKGStage1, HierarchicalStage1Config; from partcat_hkg.pra_aog import HierarchicalPRAAOGParser; print("torch", torch.__version__); print("cuda", torch.cuda.is_available()); print(HierarchicalStage1Config())' && (nvidia-smi || true)

## 2. Tests and dataset config


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && pytest -q tests/test_bidir_stage1.py tests/test_hier_pra_aog.py tests/test_pra_aog.py tests/test_strict_aog_core.py

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python -c 'from pathlib import Path; import os; p=Path(os.environ["CONFIG"]); p.write_text("extends: stage1_quality_upgrade.yaml\npaths:\n  partimagenet_root: " + repr(os.environ["PARTIMAGENET_ROOT"]) + "\n", encoding="utf-8"); print(p.read_text())' && for p in "$PARTIMAGENET_ROOT/annotations/train/train.json" "$PARTIMAGENET_ROOT/annotations/val/val.json" "$PARTIMAGENET_ROOT/images/train" "$PARTIMAGENET_ROOT/images/val" "$BASE_STAGE1_CKPT"; do if [ -e "$p" ]; then echo "FOUND $p"; else echo "MISSING $p"; fi; done

## 3. Train bidirectional hierarchical Stage 1


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python scripts/run_bidir_stage1.py --config "$CONFIG" --partimagenet-root "$PARTIMAGENET_ROOT" --save-dir "${BIDIR_STAGE1_DIR}_smoke" --warm-start "$BASE_STAGE1_CKPT" --allow-partial-load --epochs 1 --batch-size 2 --max-train-samples 8 --max-val-samples 8 --num-workers 0 --subparts-per-part 4 --feedback-weight 0.35 --smoke-only

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python scripts/run_bidir_stage1.py --config "$CONFIG" --partimagenet-root "$PARTIMAGENET_ROOT" --save-dir "$BIDIR_STAGE1_DIR" --warm-start "$BASE_STAGE1_CKPT" --allow-partial-load --epochs 20 --batch-size 16 --num-workers 4 --subparts-per-part 4 --feedback-weight 0.35 --subpart-bce 0.35 --subpart-dice 0.35 --inside-parent 0.20 --parent-cover 0.15

## 4. Cache refined terminals


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && test -f "$BIDIR_STAGE1_CKPT" && mkdir -p "$BIDIR_CACHE_DIR" && python scripts/cache_bidir_stage1_terminals.py --config "$CONFIG" --stage1-ckpt "$BIDIR_STAGE1_CKPT" --out-dir "$BIDIR_CACHE_DIR" --device auto --splits train,val --batch-size 16 --num-workers 2 --threshold 0.40 --support-gate-mode post --support-component-mode best --max-terminals 32 --mask-size 64 --shard-size 1024 --store-images --store-images-splits val --allow-partial-load && ls -lh "$TRAIN_CACHE" "$VAL_CACHE"

## 5. Build, train, and evaluate hierarchical PRA-AOG


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && mkdir -p "$HIER_BUNDLE_DIR" && python scripts/build_hier_pra_aog.py --cache "$TRAIN_CACHE" --out "$HIER_BUNDLE" --num-templates-per-class 4 --max-slots-per-template 14 --max-slots-per-part 4 --min-template-support 2 --min-slot-support 0.10 --required-tau 0.50 --min-role-overlap 0.0 --min-edge-support 0.06 --min-edge-count 2 --min-edge-information-gain 0.02 --max-edges-per-template 24 --relation-var-floor 0.006 --geom-var-floor 0.004 --count-max 6 --motif-min-references 2 --motif-min-utility 0.0 --motif-mdl-penalty 0.01 --motif-shrinkage 0.25 --subpart-grid-size 2 --subpart-min-cell-coverage 0.08 --subpart-min-support 8 --subpart-max-per-part 8 --subpart-score-boost 0.35

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog.py --bundle "$HIER_BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "${HIER_RUN_DIR}_smoke" --device auto --batch-size 4 --epochs 1 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.35 --num-workers 0 --max-train-batches 2 --max-val-batches 2

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog.py --bundle "$HIER_BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$HIER_RUN_DIR" --device auto --batch-size 16 --epochs 20 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.35 --preload-cache --num-workers 0

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && test -f "$HIER_BEST_CKPT" && python scripts/run_hier_pra_aog.py --bundle "$HIER_BUNDLE" --train-cache "$TRAIN_CACHE" --val-cache "$VAL_CACHE" --save-dir "$HIER_RUN_DIR" --checkpoint "$HIER_BEST_CKPT" --eval-only --device auto --batch-size 16 --assignment gpu_mf --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.35 --preload-cache --num-workers 0

## 6. Decode one example and package outputs


In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && cd "$REPO_DIR" && python scripts/infer_hier_pra_aog.py --bundle "$HIER_BUNDLE" --cache "$VAL_CACHE" --checkpoint "$HIER_BEST_CKPT" --out-dir "$INFER_DIR" --sample-index 0 --device auto --assignment gpu_mf --top-k 5 --posterior-tau 0.75 --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --posterior-logits --subpart-score-weight 0.35 && python -m json.tool "$INFER_DIR/sample_00000_summary.json" | head -n 260

In [ ]:
!. /tmp/bidir_hier_pra_aog_v5.env && tar -czf "$PACKAGE_OUT" -C "$(dirname "$BIDIR_STAGE1_DIR")" "$(basename "$BIDIR_STAGE1_DIR")" -C "$(dirname "$BIDIR_CACHE_DIR")" "$(basename "$BIDIR_CACHE_DIR")" -C "$(dirname "$HIER_BUNDLE_DIR")" "$(basename "$HIER_BUNDLE_DIR")" -C "$(dirname "$HIER_RUN_DIR")" "$(basename "$HIER_RUN_DIR")" && ls -lh "$PACKAGE_OUT"